# Experiment 2, Run 2: Memory Integrity Under Repeated Updating

This is a **corrected implementation run**, not a redesigned experiment.

The frozen Sarah statements, checkpoints, probes, ground truth, research questions and hypotheses are unchanged from Run 1.

Implementation corrections only:

1. Atomic routing is fixed by the preregistered protocol.
2. The LLM cannot choose affected record IDs.
3. Record IDs are assigned deterministically by code.
4. AP provenance and supersession metadata are assigned deterministically by code.
5. Unrelated records cannot modify M1 through M5.
6. Holistic rewriting has a larger generation allowance.
7. Any generation that reaches its output ceiling causes a hard failure.
8. Raw outputs remain unscored for later manual evaluation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U "transformers>=4.45" accelerate sentencepiece
import torch, gc, json, copy
from pathlib import Path
if not torch.cuda.is_available():
    raise RuntimeError("Enable a T4 GPU in Runtime settings first.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID="Qwen/Qwen2.5-1.5B-Instruct"
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True
)
model.eval()

OUT=Path("/content/drive/MyDrive/Sarah_Memory_Experiment_2_Run2")
OUT.mkdir(parents=True, exist_ok=True)
print("Permanent output:", OUT)


In [ ]:
TURNS=[(1, 'When I am considering a research collaboration and say I need more evidence, do not interpret my delay as lack of interest.'), (2, 'When I ask for a practical recommendation, give me one primary recommendation unless I explicitly say I am exploring possibilities.'), (3, 'If I delay registering for a large conference, cost is usually the first factor I want you to consider, not lack of interest.'), (4, 'When I postpone a creative project, ask what is blocking it before suggesting that my motivation has changed.'), (5, 'When we discuss an unexpected scientific result, give me at least two competing explanations before favoring one.'), (6, 'I prefer short meeting agendas.'), (7, 'For free conferences, cost should obviously not be considered as the reason for delayed registration.'), (8, 'I prefer morning research sessions.'), (9, 'For simple factual questions, I do not need competing explanations.'), (10, 'I prefer concise presentation titles.'), (11, 'I want to update what I said about research collaborations. Once I explicitly tell you that I have enough evidence, continued delay can be treated as information that something else may be holding up my decision. Do not assume what that reason is.'), (12, 'I prefer written preparation before meetings.'), (13, 'For creative projects, if I explicitly tell you that I have lost motivation, you do not need to ask what is blocking me before responding.'), (14, 'I prefer research notes organized by question.'), (15, 'I want to refine my recommendation preference. When I explicitly say I am exploring possibilities, give me exactly two contrasting options.'), (16, 'I prefer short conference follow ups.'), (17, 'I want to refine the conference rule. Cost should be considered specifically for expensive large conferences. If the conference is free, do not use cost as the explanation.'), (18, 'I prefer tables for direct comparisons.'), (19, 'One more update about research collaboration delays. If I explicitly tell you the reason for my delay, use that reason rather than treating the cause as unknown.'), (20, 'I prefer experiment conclusions separated from hypotheses.')]
ROUTING={1: ('M1', 'research collaboration delays'), 2: ('M2', 'practical recommendations'), 3: ('M3', 'large conference registration'), 4: ('M4', 'creative projects'), 5: ('M5', 'scientific reasoning'), 6: ('U6', 'meeting agendas'), 7: ('M3', 'large conference registration'), 8: ('U8', 'research session timing'), 9: ('M5', 'scientific reasoning'), 10: ('U10', 'presentation titles'), 11: ('M1', 'research collaboration delays'), 12: ('U12', 'meeting preparation'), 13: ('M4', 'creative projects'), 14: ('U14', 'research note organization'), 15: ('M2', 'practical recommendations'), 16: ('U16', 'conference follow ups'), 17: ('M3', 'large conference registration'), 18: ('U18', 'comparison formatting'), 19: ('M1', 'research collaboration delays'), 20: ('U20', 'experiment writing')}
CHECKPOINTS=[5,10,15,20]
MEMORY_PROBES={'M1': 'What do you currently know about how Sarah wants delays around research collaborations interpreted?', 'M2': 'What do you currently know about how Sarah wants practical recommendations presented?', 'M3': 'What do you currently know about how Sarah wants delayed registration for large conferences interpreted?', 'M4': 'What do you currently know about how Sarah wants postponed creative projects handled?', 'M5': 'What do you currently know about how Sarah wants unexpected scientific results reasoned about?'}
BEHAVIOR_PROBES={'M1': "I have enough evidence that a research collaboration is worthwhile. I haven't replied for eight days because I am waiting until after an important meeting tomorrow. What can you conclude from my delay?", 'M2': "I'm exploring possibilities for how to communicate a new research result. What would you suggest?", 'M3': "Registration for a large conference is free this year. I have postponed registering twice. Does that mean I'm probably not interested?", 'M4': "I've postponed my film project again. I haven't said that I've lost motivation. What do you think?", 'M5': 'An adaptive AI system performed significantly better after repeated interactions. What might explain the improvement?'}
ADVERSARIAL_PROBES={'B1': 'I have delayed replying to a friend. Does that mean I need more evidence?', 'B2': "I'm buying a 20 euro book and haven't decided yet. Should we assume cost is the main issue?", 'B3': 'What is the capital of Finland? Please give me two competing explanations.'}
GROUND_TRUTH={5: {'M1': 'Research collaboration plus need for more evidence: delay must not be interpreted as lack of interest.', 'M2': 'Practical recommendation: one primary recommendation unless Sarah explicitly says she is exploring possibilities.', 'M3': 'Delayed registration for a large conference: consider cost first rather than lack of interest.', 'M4': 'Postponed creative project: ask what is blocking it before suggesting motivation changed.', 'M5': 'Unexpected scientific result: give at least two competing explanations before favoring one.'}, 10: {'M1': 'Research collaboration plus need for more evidence: delay must not be interpreted as lack of interest.', 'M2': 'Practical recommendation: one primary recommendation unless Sarah explicitly says she is exploring possibilities.', 'M3': 'Large conference delay: consider cost first, except when the conference is free.', 'M4': 'Postponed creative project: ask what is blocking it before suggesting motivation changed.', 'M5': 'Unexpected scientific result: at least two competing explanations before favoring one. Simple factual questions do not require competing explanations.'}, 15: {'M1': 'If Sarah still needs evidence, delay must not be interpreted as lack of interest. If Sarah explicitly has enough evidence, continued delay is informative that something else may be holding up the decision, but its cause remains unknown unless Sarah supplies it.', 'M2': 'Ordinary practical recommendation: one primary recommendation. Explicitly exploring possibilities: exactly two contrasting options.', 'M3': 'Large conference delay: consider cost first, except when the conference is free.', 'M4': 'Creative delay with motivation unknown: ask what is blocking it before suggesting motivation changed. If Sarah explicitly says she lost motivation, asking what blocks her first is unnecessary.', 'M5': 'Unexpected scientific result: at least two competing explanations before favoring one. Simple factual questions do not require competing explanations.'}, 20: {'M1': 'Needs more evidence: do not infer lack of interest. Enough evidence and cause not given: continued delay is informative but do not infer its cause. If Sarah explicitly gives the cause: use that stated cause.', 'M2': 'Ordinary practical recommendation: one primary recommendation. Explicitly exploring possibilities: exactly two contrasting options.', 'M3': 'Expensive large conference: consider cost first. Free conference: do not use cost as the explanation.', 'M4': 'Creative delay with motivation unknown: ask what is blocking it before suggesting motivation changed. If Sarah explicitly says motivation is gone, asking what blocks her first is unnecessary.', 'M5': 'Unexpected scientific result: at least two competing explanations before favoring one. Simple factual question: competing explanations unnecessary.'}}

MAX_MEMORY_TOKENS=1400
MAX_R_UPDATE_TOKENS=900
MAX_LOCAL_UPDATE_TOKENS=300
MAX_RESPONSE_TOKENS=180

def toklen(x):
    return len(tokenizer.encode(x, add_special_tokens=False))

def save(name,obj):
    (OUT/name).write_text(json.dumps(obj,indent=2,ensure_ascii=False),encoding="utf-8")

save("frozen_protocol_run2.json",{
    "turns":TURNS, "routing":ROUTING, "checkpoints":CHECKPOINTS,
    "memory_probes":MEMORY_PROBES, "behavior_probes":BEHAVIOR_PROBES,
    "adversarial_probes":ADVERSARIAL_PROBES, "ground_truth":GROUND_TRUTH,
    "implementation_note":"Research protocol unchanged. Run 2 corrects routing, IDs, provenance, and truncation handling only."
})
print("Frozen Run 2 protocol saved.")


In [ ]:
def generate(prompt,max_new_tokens):
    chat=tokenizer.apply_chat_template(
        [{"role":"user","content":prompt}],
        tokenize=False, add_generation_prompt=True
    )
    x=tokenizer(chat,return_tensors="pt",truncation=True,max_length=7000)
    x={k:v.to(model.device) for k,v in x.items()}
    with torch.inference_mode():
        y=model.generate(
            **x,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )
    new_ids=y[0][x["input_ids"].shape[1]:]
    generated_tokens=len(new_ids)
    ans=tokenizer.decode(new_ids,skip_special_tokens=True).strip()

    # If the model used the entire allowance, treat it as potentially truncated.
    if generated_tokens >= max_new_tokens:
        del x,y,new_ids
        gc.collect(); torch.cuda.empty_cache()
        raise RuntimeError(
            f"Generation reached token ceiling ({generated_tokens}/{max_new_tokens}). "
            "Run stopped to prevent silent truncation."
        )

    del x,y,new_ids
    gc.collect(); torch.cuda.empty_cache()
    return ans, generated_tokens

def assert_memory_budget(memory_text):
    n=toklen(memory_text)
    if n>MAX_MEMORY_TOKENS:
        raise RuntimeError(
            f"Stored memory exceeded frozen budget: {n} > {MAX_MEMORY_TOKENS}. "
            "Do not silently truncate."
        )
    return n


In [ ]:
# Fixed routing means the model never chooses which record to edit.

def update_R(memory,msg,turn):
    prompt=f"""You maintain Sarah's complete semantic memory.

Integrate Sarah's new statement into the complete memory.
Preserve every still-relevant prior rule, qualification, exception and contextual boundary.
When Sarah explicitly updates an earlier rule, revise that rule without deleting still-valid conditions.
Do not infer personality, motives, psychological traits, or unstated preferences.
Do not learn from your own prior wording.
Keep separate topics separate.
Be concise, but preserve meaning faithfully.

CURRENT COMPLETE MEMORY:
{memory or "No stored information."}

NEW EXPLICIT STATEMENT FROM SARAH AT INTERACTION {turn}:
{msg}

Return only the complete updated memory."""
    return generate(prompt,MAX_R_UPDATE_TOKENS)

def create_atomic_record(record_id,scope,msg):
    # New atomic information is stored from Sarah's explicit statement directly.
    return {"id":record_id,"scope":scope,"rule":msg,"status":"active"}

def revise_atomic_rule(old_rule,new_statement,scope):
    prompt=f"""Revise ONE atomic memory rule using Sarah's explicit update.

Scope: {scope}

CURRENT RULE:
{old_rule}

NEW EXPLICIT STATEMENT FROM SARAH:
{new_statement}

Requirements:
Preserve all still-valid parts of the current rule.
Integrate only what Sarah explicitly changed, qualified or added.
Do not infer personality, motives or unstated preferences.
Do not generalize beyond this scope.
Return only the revised rule text, with no label or commentary."""
    return generate(prompt,MAX_LOCAL_UPDATE_TOKENS)

def active_ap_record(records,base_id):
    matches=[r for r in records if r.get("base_id")==base_id and r.get("status")=="active"]
    if len(matches)!=1:
        raise RuntimeError(f"Expected exactly one active AP record for {base_id}, found {len(matches)}")
    return matches[0]

def serialize_A(records):
    ordered=[records[k] for k in sorted(records, key=lambda x:(x.startswith("U"), int(x[1:])) if x[1:].isdigit() else (False,x))]
    return json.dumps(ordered,ensure_ascii=False,sort_keys=True)

def serialize_AP(records):
    return json.dumps(records,ensure_ascii=False,sort_keys=True)


In [ ]:
def run_R():
    path=OUT/"R_states.json"
    if path.exists():
        print("Loading completed R.")
        return json.loads(path.read_text(encoding="utf-8"))

    memory=""
    log=[]
    for turn,msg in TURNS:
        memory,gen_tokens=update_R(memory,msg,turn)
        mem_tokens=assert_memory_budget(memory)
        row={"turn":turn,"statement":msg,"memory":memory,
             "memory_tokens":mem_tokens,"update_generated_tokens":gen_tokens}
        log.append(row)
        save("R_progress.json",log)
        print("R turn",turn,"saved | memory",mem_tokens,"tokens | generated",gen_tokens)
    save("R_states.json",log)
    return log

def run_A():
    path=OUT/"A_states.json"
    if path.exists():
        print("Loading completed A.")
        return json.loads(path.read_text(encoding="utf-8"))

    records={}
    log=[]
    update_log=[]
    for turn,msg in TURNS:
        rid,scope=ROUTING[turn]

        if rid not in records:
            records[rid]=create_atomic_record(rid,scope,msg)
            action="create"
            gen_tokens=0
        else:
            before=copy.deepcopy(records[rid])
            revised,gen_tokens=revise_atomic_rule(before["rule"],msg,scope)
            records[rid]={"id":rid,"scope":scope,"rule":revised,"status":"active"}
            action="revise"

        state=serialize_A(records)
        mem_tokens=assert_memory_budget(state)
        log.append({"turn":turn,"statement":msg,"routed_to":rid,
                    "action":action,"memory":state,"memory_tokens":mem_tokens,
                    "update_generated_tokens":gen_tokens})
        update_log.append({"turn":turn,"routed_to":rid,"action":action,
                           "record_after":copy.deepcopy(records[rid])})
        save("A_progress.json",log)
        save("A_updates.json",update_log)
        print("A turn",turn,"->",rid,action,"| memory",mem_tokens,"tokens")
    save("A_states.json",log)
    return log

def run_AP():
    path=OUT/"AP_states.json"
    if path.exists():
        print("Loading completed AP.")
        return json.loads(path.read_text(encoding="utf-8"))

    records=[]
    versions={}
    log=[]
    update_log=[]

    for turn,msg in TURNS:
        base_id,scope=ROUTING[turn]
        versions.setdefault(base_id,0)

        existing=[r for r in records if r.get("base_id")==base_id and r.get("status")=="active"]

        if not existing:
            versions[base_id]+=1
            vid=f"{base_id}_v{versions[base_id]}"
            newrec={
                "id":vid,"base_id":base_id,"scope":scope,"rule":msg,
                "status":"active","source":"Sarah","source_interaction":turn,
                "evidence_type":"explicit statement","supersedes":None
            }
            records.append(newrec)
            action="create"
            gen_tokens=0
            prior_id=None
        else:
            if len(existing)!=1:
                raise RuntimeError(f"AP routing integrity error for {base_id}: {len(existing)} active records")
            old=existing[0]
            revised,gen_tokens=revise_atomic_rule(old["rule"],msg,scope)
            old["status"]="superseded"

            versions[base_id]+=1
            vid=f"{base_id}_v{versions[base_id]}"
            prior_id=old["id"]
            newrec={
                "id":vid,"base_id":base_id,"scope":scope,"rule":revised,
                "status":"active","source":"Sarah","source_interaction":turn,
                "evidence_type":"explicit statement","supersedes":prior_id
            }
            records.append(newrec)
            action="revise"

        state=serialize_AP(records)
        mem_tokens=assert_memory_budget(state)
        log.append({"turn":turn,"statement":msg,"routed_to":base_id,
                    "action":action,"memory":state,"memory_tokens":mem_tokens,
                    "update_generated_tokens":gen_tokens})
        update_log.append({"turn":turn,"routed_to":base_id,"action":action,
                           "new_active_id":vid,"supersedes":prior_id})
        save("AP_progress.json",log)
        save("AP_updates.json",update_log)
        print("AP turn",turn,"->",base_id,action,"| memory",mem_tokens,"tokens")

    save("AP_states.json",log)
    return log

states={"R":run_R(),"A":run_A(),"AP":run_AP()}
print("All three memory conditions complete.")


In [ ]:
# Verify architectural invariants before probing.
def parse_state(c,row):
    if c=="R":
        return row["memory"]
    return json.loads(row["memory"])

# A: every target ID exists after introduction and unrelated turns cannot delete it.
for row in states["A"]:
    recs={r["id"]:r for r in json.loads(row["memory"])}
    introduced=[ROUTING[t][0] for t,_ in TURNS if t<=row["turn"]]
    for rid in set(introduced):
        if rid not in recs:
            raise RuntimeError(f"A invariant failed at turn {row['turn']}: missing {rid}")

# AP: exactly one active version exists for every introduced base ID.
for row in states["AP"]:
    recs=json.loads(row["memory"])
    introduced=set(ROUTING[t][0] for t,_ in TURNS if t<=row["turn"])
    for bid in introduced:
        active=[r for r in recs if r["base_id"]==bid and r["status"]=="active"]
        if len(active)!=1:
            raise RuntimeError(f"AP invariant failed at turn {row['turn']} for {bid}")

print("Architectural invariant checks passed.")


In [ ]:
SYSTEM="""You are an AI assistant helping Sarah.
Use the supplied memory only when relevant.
Respect contextual boundaries, exceptions, and superseded information.
Do not invent preferences, motives, or traits."""

def checkpoint_state(c,turn):
    return next(x for x in states[c] if x["turn"]==turn)["memory"]

# Save checkpoint states before any probes.
checkpoints={}
for c in ["R","A","AP"]:
    checkpoints[c]={str(cp):{
        "memory":checkpoint_state(c,cp),
        "memory_tokens":next(x for x in states[c] if x["turn"]==cp)["memory_tokens"]
    } for cp in CHECKPOINTS}
save("checkpoint_states.json",checkpoints)
print("Checkpoint states frozen.")


In [ ]:
memory_rows=[]
for c in ["R","A","AP"]:
    for cp in CHECKPOINTS:
        memory=checkpoint_state(c,cp)
        for mid,q in MEMORY_PROBES.items():
            prompt=f"""{SYSTEM}

MEMORY ABOUT SARAH:
{memory}

MEMORY INSPECTION QUESTION:
{q}

Report only what the supplied memory supports. Do not add assumptions."""
            ans,n=generate(prompt,MAX_RESPONSE_TOKENS)
            memory_rows.append({"condition":c,"checkpoint":cp,"memory_id":mid,
                                "question":q,"answer":ans,"generated_tokens":n})
            save("memory_probe_responses.json",memory_rows)
        print(c,"checkpoint",cp,"memory probes saved.")


In [ ]:
behavior_rows=[]
for c in ["R","A","AP"]:
    for cp in CHECKPOINTS:
        memory=checkpoint_state(c,cp)
        for mid,q in BEHAVIOR_PROBES.items():
            prompt=f"""{SYSTEM}

MEMORY ABOUT SARAH:
{memory}

SARAH'S CURRENT MESSAGE:
{q}

Respond directly to Sarah."""
            ans,n=generate(prompt,MAX_RESPONSE_TOKENS)
            behavior_rows.append({"condition":c,"checkpoint":cp,"memory_id":mid,
                                  "question":q,"answer":ans,"generated_tokens":n})
            save("behavior_probe_responses.json",behavior_rows)
        print(c,"checkpoint",cp,"behavior probes saved.")


In [ ]:
boundary_rows=[]
for c in ["R","A","AP"]:
    memory=checkpoint_state(c,20)
    for pid,q in ADVERSARIAL_PROBES.items():
        prompt=f"""{SYSTEM}

MEMORY ABOUT SARAH:
{memory}

SARAH'S CURRENT MESSAGE:
{q}

Respond directly to Sarah. Apply stored preferences only within their supported scope."""
        ans,n=generate(prompt,MAX_RESPONSE_TOKENS)
        boundary_rows.append({"condition":c,"checkpoint":20,"probe_id":pid,
                              "question":q,"answer":ans,"generated_tokens":n})
        save("boundary_probe_responses.json",boundary_rows)

manifest={
 "run":"Experiment 2 Run 2",
 "model":MODEL_ID,
 "generation":{
   "do_sample":False,
   "max_memory_tokens":MAX_MEMORY_TOKENS,
   "max_R_update_tokens":MAX_R_UPDATE_TOKENS,
   "max_local_update_tokens":MAX_LOCAL_UPDATE_TOKENS,
   "max_response_tokens":MAX_RESPONSE_TOKENS
 },
 "implementation_corrections":[
   "Fixed protocol routing for A and AP",
   "Deterministic record IDs",
   "Deterministic AP provenance and supersession metadata",
   "Unrelated records cannot alter M1-M5",
   "Hard failure on generation ceiling",
   "Larger holistic rewrite allowance",
   "Architectural invariant checks before probes"
 ],
 "scoring":"No automatic scoring performed."
}
save("manifest.json",manifest)

import shutil
zip_path=shutil.make_archive(
    "/content/drive/MyDrive/Sarah_Memory_Experiment_2_Run2",
    "zip",root_dir=str(OUT)
)
print("Complete. ZIP saved at:",zip_path)

from google.colab import files
files.download(zip_path)
